# 02 ? Ti?n x? l? d? li?u

Notebook n?y ki?m tra d? li?u v? m? t? b??c ti?n x? l? ???c d?ng khi hu?n luy?n. Kh?ng ?i?n median tr?n to?n b? dataset ? ??y: median ???c h?c b?n trong `Pipeline` t? t?ng fold train, tr?nh r? r? d? li?u. Quy tr?nh ch?nh th?c n?m ? `03_train.ipynb` v? `src/train.py.

In [ ]:
import io
import zipfile
import pandas as pd
try:
    from google.colab import files
except ModuleNotFoundError:
    files = None

def read_csv_from_zip(zip_bytes):
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as archive:
        csv_name = next(name for name in archive.namelist() if name.lower().endswith('.csv'))
        with archive.open(csv_name) as csv_file:
            return pd.read_csv(csv_file)

if files is not None:
    uploaded = files.upload()
    filename = next(iter(uploaded))
    df = read_csv_from_zip(uploaded[filename]) if filename.lower().endswith('.zip') else pd.read_csv(io.BytesIO(uploaded[filename]))
else:
    from pathlib import Path
    roots = [Path.cwd(), *Path.cwd().parents]
    ai_models_dir = next((p for root in roots for p in (root, root / 'ai-models') if p.name == 'ai-models' and p.is_dir()), None)
    if ai_models_dir is None:
        raise FileNotFoundError('Kh?ng t?m th?y ai-models/.')
    df = pd.read_csv(ai_models_dir / 'data' / 'diabetes.csv')

print('K?ch th??c:', df.shape)
display(df.head())
display(df.dtypes.rename('dtype').to_frame())

## Ki?m tra gi? tr? thi?u v? s? 0

Theo pipeline train, s? 0 ???c xem l? gi? tr? kh?ng h?p l? ? `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, `BMI`. C?c c?t n?y ???c thay b?ng median trong pipeline; nh?ng c?t kh?c gi? nguy?n. `Outcome` l? nh?n, kh?ng ph?i feature.

In [ ]:
invalid_zero_columns = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
feature_columns = [c for c in df.columns if c not in ('Outcome', 'index')]
missing_summary = df[feature_columns].isna().sum().rename('missing')
zero_summary = df[[c for c in invalid_zero_columns if c in df.columns]].eq(0).sum().rename('invalid_zero')
display(pd.concat([missing_summary, zero_summary], axis=1).fillna(0).astype(int))
print('S? l??ng nh?n Outcome:')
display(df['Outcome'].value_counts(dropna=False).sort_index())

### Ti?n x? l? khi train

`03_train.ipynb` d?ng `ColumnTransformer(SimpleImputer(strategy="median", missing_values=0))`, sau ?? `StandardScaler`, r?i classifier. T?t c? b??c n?m trong `Pipeline` n?n m?i fold ch? fit imputer/scaler tr?n ph?n train c?a fold. Kh?ng c?n l?u m?t dataset ?? scale ri?ng; inference d?ng c?ng pipeline ?? fit.